# 演算法偏見與公平性

## 📌 學習目標

完成本 Notebook 後，你將能夠：

1. 說明資料偏見與模型偏見的常見來源。
2. 使用 Python 計算常見公平性指標，例如 Demographic Parity、Equal Opportunity、Equalized Odds 與 Disparate Impact。
3. 觀察同一個模型在不同群體上的預測差異。
4. 嘗試透過重新抽樣與決策門檻調整，降低演算法偏見。

本練習以一個簡化的「貸款核准」情境示範公平性評估。資料為教學用合成資料，不代表真實金融決策。


In [ ]:
# ── 環境設定 ────────────────────────────────────
# 載入本章節所需的 Python 套件，並建立可重現的隨機種子。

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.utils import resample

np.random.seed(42)

print('環境設定完成')
print('可使用套件：numpy、pandas、matplotlib、sklearn')


## 核心概念說明

演算法偏見通常不是單一原因造成，而是資料、模型目標與部署情境共同作用的結果。

### 1. 資料偏見

資料若沒有充分代表所有群體，模型就可能學到偏頗規則。例如訓練資料過度集中在特定年齡、收入、地區或性別群體，會造成來源偏誤；若資料本身包含歷史刻板印象，則可能形成內容偏誤；若標註者的判斷不一致，也可能形成製程偏誤。

### 2. 模型偏見

即使資料看似合理，模型也可能因為只追求整體準確率，而犧牲少數群體的表現。常見情況包括目標函數偏誤、演算法偏見、正規化導致少數群體特徵被簡化等。

### 3. 常見公平性指標

- Demographic Parity：不同群體獲得正向預測的比例是否接近。
- Equal Opportunity：在真實正例中，不同群體被正確預測為正向的比例是否接近。
- Equalized Odds：不同群體的 TPR 與 FPR 是否都接近。
- Disparate Impact：弱勢群體正向結果比例是否至少達參照群體的 80%。


In [ ]:
# ── 示範：建立含有偏見風險的合成資料 ────────────────────────
# 這段程式碼建立一份教學用貸款資料，其中不同群體的收入分布與歷史核准機率略有差異，用來模擬資料代表性與歷史偏誤。

import numpy as np
import pandas as pd

np.random.seed(42)
n = 1200

# group=0 代表 A 群體，group=1 代表 B 群體
# 教學情境中，B 群體在歷史資料中平均收入較低，且歷史核准條件較不利
group = np.random.binomial(1, 0.45, n)
age = np.random.normal(38, 9, n).clip(20, 65)
income = np.where(group == 0,
                  np.random.normal(72, 18, n),
                  np.random.normal(58, 16, n)).clip(20, 130)
credit_score = np.where(group == 0,
                        np.random.normal(680, 55, n),
                        np.random.normal(650, 60, n)).clip(450, 850)
debt_ratio = np.random.beta(2, 5, n)

# 真實是否適合核准：主要由還款能力決定
true_score = 0.035 * income + 0.009 * (credit_score - 600) - 2.2 * debt_ratio + 0.01 * (age - 35)
prob_qualified = 1 / (1 + np.exp(-true_score + 2.3))
qualified = np.random.binomial(1, prob_qualified)

# 歷史標籤含有不公平因素：B 群體即使條件相近，也較不容易被核准
historical_score = true_score - 0.55 * group
prob_approved = 1 / (1 + np.exp(-historical_score + 2.3))
approved = np.random.binomial(1, prob_approved)

df = pd.DataFrame({
    'group': group,
    'age': age.round(1),
    'income': income.round(1),
    'credit_score': credit_score.round(0),
    'debt_ratio': debt_ratio.round(3),
    'qualified': qualified,
    'approved': approved
})

display(df.head())
print('各群體筆數：')
print(df['group'].value_counts().rename(index={0: 'A群體', 1: 'B群體'}))
print('\n歷史核准率：')
print(df.groupby('group')['approved'].mean().rename(index={0: 'A群體', 1: 'B群體'}).round(3))


## 公平性指標的實作重點

在公平性分析中，我們通常會同時觀察模型效能與群體差異。

- 正向預測率：模型預測為核准的比例，可用於 Demographic Parity。
- TPR：真實應核准者中，被模型正確核准的比例，可用於 Equal Opportunity。
- FPR：真實不應核准者中，被模型錯誤核准的比例，是 Equalized Odds 的一部分。
- Disparate Impact：B 群體正向預測率除以 A 群體正向預測率，若低於 0.8，通常代表需要進一步檢視是否存在不利影響。

注意：公平性指標沒有唯一最佳答案。不同應用場景會重視不同的公平性定義，例如醫療篩檢可能更重視 Equal Opportunity，招聘或入學機會可能更重視 Demographic Parity。


In [ ]:
# ── 示範：訓練模型並計算公平性指標 ─────────────────────────
# 這段程式碼訓練一個邏輯斯迴歸模型，並比較不同群體的正向預測率、TPR、FPR 與不利影響比。

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

np.random.seed(42)
n = 1200
group = np.random.binomial(1, 0.45, n)
age = np.random.normal(38, 9, n).clip(20, 65)
income = np.where(group == 0, np.random.normal(72, 18, n), np.random.normal(58, 16, n)).clip(20, 130)
credit_score = np.where(group == 0, np.random.normal(680, 55, n), np.random.normal(650, 60, n)).clip(450, 850)
debt_ratio = np.random.beta(2, 5, n)
true_score = 0.035 * income + 0.009 * (credit_score - 600) - 2.2 * debt_ratio + 0.01 * (age - 35)
qualified = np.random.binomial(1, 1 / (1 + np.exp(-true_score + 2.3)))
historical_score = true_score - 0.55 * group
approved = np.random.binomial(1, 1 / (1 + np.exp(-historical_score + 2.3)))

df = pd.DataFrame({
    'group': group,
    'age': age,
    'income': income,
    'credit_score': credit_score,
    'debt_ratio': debt_ratio,
    'qualified': qualified,
    'approved': approved
})

X = df[['age', 'income', 'credit_score', 'debt_ratio', 'group']]
y = df['approved']
X_train, X_test, y_train, y_test, q_train, q_test = train_test_split(
    X, y, df['qualified'], test_size=0.3, random_state=7, stratify=df['group']
)

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

result = X_test.copy()
result['y_true_historical'] = y_test.values
result['qualified'] = q_test.values
result['prediction'] = y_pred

summary = []
for g, label in [(0, 'A群體'), (1, 'B群體')]:
    part = result[result['group'] == g]
    positive_rate = part['prediction'].mean()
    tpr = part.loc[part['qualified'] == 1, 'prediction'].mean()
    fpr = part.loc[part['qualified'] == 0, 'prediction'].mean()
    summary.append([label, len(part), positive_rate, tpr, fpr])

metrics = pd.DataFrame(summary, columns=['群體', '樣本數', '正向預測率', 'TPR', 'FPR'])
di = metrics.loc[metrics['群體'] == 'B群體', '正向預測率'].iloc[0] / metrics.loc[metrics['群體'] == 'A群體', '正向預測率'].iloc[0]

print('整體準確率：', round(accuracy_score(y_test, y_pred), 3))
display(metrics.round(3))
print('Disparate Impact（B群體 / A群體）：', round(di, 3))
print('是否通過 80% Rule：', '通過' if di >= 0.8 else '未通過，需要進一步檢視')


## 降低演算法偏見的方法

當模型呈現群體差異時，可以從不同階段介入：

### 1. 資料前處理

在訓練前調整資料，例如重新抽樣、補足少數群體樣本、移除或泛化敏感特徵。優點是容易導入既有流程；限制是可能無法處理模型訓練時產生的偏見。

### 2. 模型內處理

在訓練過程加入公平性約束，讓模型同時考量準確率與公平性。優點是可直接影響學習目標；限制是實作較複雜，也可能犧牲部分效能。

### 3. 結果後處理

模型訓練完成後，調整不同群體的決策門檻，讓重要公平性指標更接近。優點是不用重新訓練模型；限制是需要謹慎評估合規性與業務合理性。

下面的實作示範兩種簡化方法：重新抽樣與群體門檻調整。


In [ ]:
# ── 實際應用：重新抽樣與門檻調整 ──────────────────────────
# 這段程式碼示範兩種常見去偏思路：先用重新抽樣平衡訓練資料，再用群體門檻調整改善正向預測率差異。

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.utils import resample

np.random.seed(42)
n = 1200
group = np.random.binomial(1, 0.45, n)
age = np.random.normal(38, 9, n).clip(20, 65)
income = np.where(group == 0, np.random.normal(72, 18, n), np.random.normal(58, 16, n)).clip(20, 130)
credit_score = np.where(group == 0, np.random.normal(680, 55, n), np.random.normal(650, 60, n)).clip(450, 850)
debt_ratio = np.random.beta(2, 5, n)
true_score = 0.035 * income + 0.009 * (credit_score - 600) - 2.2 * debt_ratio + 0.01 * (age - 35)
qualified = np.random.binomial(1, 1 / (1 + np.exp(-true_score + 2.3)))
historical_score = true_score - 0.55 * group
approved = np.random.binomial(1, 1 / (1 + np.exp(-historical_score + 2.3)))

df = pd.DataFrame({
    'group': group,
    'age': age,
    'income': income,
    'credit_score': credit_score,
    'debt_ratio': debt_ratio,
    'qualified': qualified,
    'approved': approved
})

X = df[['age', 'income', 'credit_score', 'debt_ratio', 'group']]
y = df['approved']
X_train, X_test, y_train, y_test, q_train, q_test = train_test_split(
    X, y, df['qualified'], test_size=0.3, random_state=7, stratify=df['group']
)

train_df = X_train.copy()
train_df['approved'] = y_train.values

# 重新抽樣：讓 A、B 群體在訓練資料中的筆數接近
train_a = train_df[train_df['group'] == 0]
train_b = train_df[train_df['group'] == 1]
target_size = max(len(train_a), len(train_b))
train_a_bal = resample(train_a, replace=True, n_samples=target_size, random_state=1)
train_b_bal = resample(train_b, replace=True, n_samples=target_size, random_state=1)
train_balanced = pd.concat([train_a_bal, train_b_bal], axis=0)

model = LogisticRegression(max_iter=1000)
model.fit(train_balanced[['age', 'income', 'credit_score', 'debt_ratio', 'group']], train_balanced['approved'])

proba = model.predict_proba(X_test)[:, 1]

# 後處理：B 群體使用略低門檻，觀察正向率與 DI 是否改善
pred_default = (proba >= 0.50).astype(int)
pred_adjusted = np.where((X_test['group'].values == 1) & (proba >= 0.43), 1, (proba >= 0.50).astype(int))

def fairness_table(pred, name):
    temp = X_test.copy()
    temp['prediction'] = pred
    temp['qualified'] = q_test.values
    rows = []
    for g, label in [(0, 'A群體'), (1, 'B群體')]:
        part = temp[temp['group'] == g]
        rows.append({
            '方案': name,
            '群體': label,
            '正向預測率': part['prediction'].mean(),
            'TPR': part.loc[part['qualified'] == 1, 'prediction'].mean(),
            'FPR': part.loc[part['qualified'] == 0, 'prediction'].mean()
        })
    return pd.DataFrame(rows)

def disparate_impact(table):
    a = table.loc[table['群體'] == 'A群體', '正向預測率'].iloc[0]
    b = table.loc[table['群體'] == 'B群體', '正向預測率'].iloc[0]
    return b / a

t1 = fairness_table(pred_default, '重新抽樣')
t2 = fairness_table(pred_adjusted, '重新抽樣＋門檻調整')
report = pd.concat([t1, t2], ignore_index=True)

display(report.round(3))
print('重新抽樣 DI：', round(disparate_impact(t1), 3))
print('重新抽樣＋門檻調整 DI：', round(disparate_impact(t2), 3))
print('重新抽樣準確率：', round(accuracy_score(y_test, pred_default), 3))
print('門檻調整後準確率：', round(accuracy_score(y_test, pred_adjusted), 3))


In [ ]:
# ── 🧪 自我測驗 ──────────────────────────────────
# 請完成下方 TODO 填空，實作公平性指標計算函式，並判斷是否通過 80% Rule。

import numpy as np
import pandas as pd

# 題目：請計算 A、B 兩群體的正向預測率與 Disparate Impact。
# TODO 1：將 positive_rate_a 填成 A 群體 prediction 的平均值。
# TODO 2：將 positive_rate_b 填成 B 群體 prediction 的平均值。
# TODO 3：將 disparate_impact 填成 positive_rate_b / positive_rate_a。

sample = pd.DataFrame({
    'group': [0, 0, 0, 0, 1, 1, 1, 1],
    'prediction': [1, 1, 0, 1, 1, 0, 0, 0]
})

positive_rate_a = sample.loc[sample['group'] == 0, 'prediction'].mean()  # TODO 1
positive_rate_b = sample.loc[sample['group'] == 1, 'prediction'].mean()  # TODO 2
disparate_impact = positive_rate_b / positive_rate_a  # TODO 3

print('A群體正向預測率 =', round(positive_rate_a, 3))
print('B群體正向預測率 =', round(positive_rate_b, 3))
print('Disparate Impact =', round(disparate_impact, 3))
print('80% Rule 判斷 =', '通過' if disparate_impact >= 0.8 else '未通過')

# Expected:
# A群體正向預測率 = 0.75
# B群體正向預測率 = 0.25
# Disparate Impact = 0.333
# 80% Rule 判斷 = 未通過
